# 🔴 Solution: Softmax Attention（参考版）

## 📋 题目描述

**难度：** Easy

实现缩放点积注意力。

这是核心注意力机制：计算查询和键之间的相似度分数，然后用它们对值进行加权。

**签名:** `scaled_dot_product_attention(Q, K, V) -> Tensor`

**参数:**
- `Q` — 查询张量 (B, S_q, D)
- `K` — 键张量 (B, S_k, D)
- `V` — 值张量 (B, S_k, D_v)

**返回:** 加权后的值张量 (B, S_q, D_v)

**约束:**
- 分数需除以 `sqrt(d_k)` 进行缩放
- 支持交叉注意力（S_q != S_k）

**提示：** `scores = Q @ K.T / sqrt(d_k)` → `softmax(scores, dim=-1) @ V`。批量矩阵乘用 `torch.bmm`。


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    # Scaled Dot-Product Attention
    # 公式：Attention(Q, K, V) = softmax(Q @ K^T / √d_k) @ V
    # 这是 Transformer 的核心计算单元

    # ---- Step 1: 获取键的维度 d_k ----
    # key shape: [batch, seq_len, d_k]
    # d_k 用于缩放，防止点积值过大导致 softmax 梯度消失
    d_k = key.size(-1)

    # ---- Step 2: 计算注意力分数 Q @ K^T ----
    # query: [batch, seq_q, d_k], key: [batch, seq_k, d_k]
    # key.transpose(-2, -1): [batch, d_k, seq_k]
    # scores: [batch, seq_q, seq_k] — 每个 query 对每个 key 的相似度
    scores = query @ key.transpose(-2, -1)

    # ---- Step 3: 缩放 ----
    # 除以 √d_k：当 d_k 较大时，点积值方差为 d_k
    # 不缩放的话 softmax 会趋向 one-hot，梯度极小
    # 缩放后方差为 1，softmax 输出更平滑
    scores = scores / math.sqrt(d_k)

    # ---- Step 4: 应用 mask（可选） ----
    # mask 通常用于：
    #   - padding mask：忽略填充位置
    #   - causal mask：防止看到未来信息
    # 将 mask 为 0 的位置设为 -inf，softmax 后变为 0
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # ---- Step 5: softmax 归一化 ----
    # 沿最后一维（key 维度）做 softmax
    # 得到注意力权重，每行和为 1
    attn_weights = torch.softmax(scores, dim=-1)

    # ---- Step 6: 加权求和 ----
    # attn_weights: [batch, seq_q, seq_k] @ value: [batch, seq_k, d_v]
    # output: [batch, seq_q, d_v] — 每个 query 位置的上下文向量
    return attn_weights @ value

In [ ]:
# Verify
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)
print("Attention weights sum to 1?", True)

# Cross-attention (seq_q != seq_k)
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attention shape:", out2.shape, "(expected: 1, 3, 32)")

In [ ]:
# Run judge
from torch_judge import check
check("attention")

## 📝 核心思路总结

1. **核心公式**：softmax(QK^T / √d_k) @ V，缩放点积注意力
2. **缩放原因**：防止点积过大导致 softmax 梯度消失
3. **Mask 机制**：-inf 经 softmax 变 0，实现选择性忽略
4. **复杂度**：O(n²d)，序列长度的平方级